<div style="background:#0A005A;border-radius:18px;padding:28px 32px;color:#fff;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif">

<div style="margin-top:22px"><span style="background:#613EFF;color:#fff;padding:5px 12px;border-radius:999px;font-size:.8em;font-weight:700;letter-spacing:.06em">TUTORIAL · NOTEBOOK</span>
<span style="color:#C9C2FF;font-size:.85em;margin-left:8px">Engenharia de Dados · Nível 0 · Fundamentos</span></div>
<h1 style="color:#fff;margin:14px 0 8px;font-size:2.1em;line-height:1.15">Prática: gravar sem duplicar</h1>
<p style="color:#E4E0FF;margin:0 0 12px;font-size:1.05em">Leve os arquivos da coleta para um banco de dados e faça a carga poder rodar quantas vezes for preciso.</p>
<span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">cerca de 25 minutos</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">Python 3.10+</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">roda no Colab</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">dados fictícios</span>
</div>

## O que você vai fazer

- Criar um banco de dados num arquivo, com o DuckDB.
- Gravar as viagens numa tabela e ver a carga duplicar tudo na segunda vez.
- Corrigir a carga para substituir o dia, e não somar a ele.

A explicação completa está na aula: [Prática: gravar sem duplicar](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-fundamentos-gravar.md).

## Como usar este notebook

Rode as células **em ordem**, de cima para baixo: clique na célula e aperte <kbd>Shift</kbd> + <kbd>Enter</kbd>. A primeira linha de cada célula diz o que fazer com ela:

| Começa com | O que fazer |
|---|---|
| `# Rode e observe` | Código pronto. Rode, leia a saída e compare com a explicação acima dela. |
| `# Sua vez` | Complete o código onde está `None` e rode. |
| `# Confira` | Rode depois da sua vez: ela mostra `[OK]` se está certo, `[ERRO]` com o que falta, ou `[AVISO]` se você ainda não respondeu. |

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Onde rodar</b><br>No <b>Google Colab</b> (só abrir, sem instalar nada), no <b>VS Code</b> com a extensão Jupyter ou no <b>JupyterLab</b> (<code>pip install jupyterlab</code> e depois <code>jupyter lab</code>). Os dados são fictícios e ficam no próprio notebook: nada vai para fora da sua máquina.</div>

##  Preparação

Instale os pacotes, carregue os quadros de conferência e crie o cenário da Pedala Aurora.

In [ ]:
%pip install -q duckdb httpx pandas

In [ ]:
from IPython.display import HTML, display


def mostrar(tipo, titulo, detalhe=""):
    """Mostra um quadro colorido com o resultado de uma conferência."""
    cor, rotulo = {"ok": ("#613EFF", "[OK]"), "erro": ("#F9006F", "[ERRO]"), "info": ("#0A005A", "[AVISO]")}[tipo]
    display(HTML(
        f'<div style="border-left:5px solid {cor};background:#F2F1F6;padding:10px 14px;'
        f'border-radius:8px;margin:6px 0;color:#202020"><b style="color:{cor}">{rotulo} {titulo}</b>'
        + (f"<br>{detalhe}" if detalhe else "") + "</div>"))


def conferir(resposta, certo, mensagem_certo, mensagem_errado):
    """Compara a sua resposta com a esperada e mostra um quadro verde, rosa ou de aviso."""
    if resposta is None:
        mostrar("info", "Ainda sem resposta", "Complete a célula “Sua vez” acima, rode-a e rode esta de novo.")
    elif certo:
        mostrar("ok", mensagem_certo)
    else:
        mostrar("erro", mensagem_errado)


mostrar("ok", "Quadros de conferência prontos")

A célula abaixo gera os dados e liga uma **API de mentira** aqui mesmo, no notebook. Ela está recolhida porque você não precisa ler o código para seguir; se tiver curiosidade, abra.

In [ ]:
#@title Preparar o cenário (clique duas vezes para ver o código)
# Cria os dados da Pedala Aurora e liga, aqui mesmo no notebook, uma API de mentira que os entrega.
# Os dados são sintéticos e sempre os mesmos (semente fixa). Você não precisa ler este código para seguir.
import json
import random
import threading
from datetime import date, datetime, timedelta, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

FUSO = timezone(timedelta(hours=-3))
ESTACOES = [
    {"id_estacao": "E01", "nome": "Praça Central", "bairro": "Centro", "vagas": 20},
    {"id_estacao": "E02", "nome": "Rodoviária", "bairro": "Centro", "vagas": 15},
    {"id_estacao": "E03", "nome": "Parque das Águas", "bairro": "Jardim Norte", "vagas": 12},
    {"id_estacao": "E04", "nome": "Mercado Municipal", "bairro": "Centro", "vagas": 16},
    {"id_estacao": "E05", "nome": "Universidade", "bairro": "Vila Nova", "vagas": 18},
    {"id_estacao": "E06", "nome": "Hospital Regional", "bairro": "Vila Nova", "vagas": 10},
    {"id_estacao": "E07", "nome": "Orla do Lago", "bairro": "Lagoa", "vagas": 14},
    {"id_estacao": "E08", "nome": "Estação do Metrô", "bairro": "Jardim Sul", "vagas": 20},
]
PRIMEIRO_DIA, ULTIMO_DIA = date(2026, 6, 1), date(2026, 6, 7)
POR_PAGINA = 40


def _gerar():
    sorteio = random.Random(2026)
    pesos = [5, 3, 2, 3, 4, 1, 2, 5]
    horas = list(range(6, 23))
    peso_hora = [2, 6, 9, 5, 3, 3, 4, 3, 3, 3, 4, 7, 9, 6, 3, 2, 1]
    viagens, atrasadas, numero = {}, [], 5000
    dia = PRIMEIRO_DIA
    while dia <= ULTIMO_DIA:
        quantas = sorteio.randint(85, 115) if dia.weekday() < 5 else sorteio.randint(55, 75)
        lista = []
        for _ in range(quantas):
            numero += 1
            saida, chegada = sorteio.choices(ESTACOES, pesos, k=2)
            hora = sorteio.choices(horas, peso_hora)[0]
            inicio = datetime(dia.year, dia.month, dia.day, hora, sorteio.randint(0, 59), tzinfo=FUSO)
            fim = inicio + timedelta(minutes=sorteio.randint(3, 45))
            lista.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": saida["id_estacao"], "estacao_chegada": chegada["id_estacao"],
                          "inicio": inicio.isoformat(), "fim": fim.isoformat()})
        lista.sort(key=lambda v: v["inicio"])
        viagens[dia.isoformat()] = lista
        dia += timedelta(days=1)
    # Defeitos plantados, como os de uma fonte real:
    for v in viagens["2026-06-04"][10:12]:                 # fim antes do início
        v["inicio"], v["fim"] = v["fim"], v["inicio"]
    for dia_, i in (("2026-06-02", 30), ("2026-06-04", 50), ("2026-06-06", 20)):
        viagens[dia_][i]["estacao_saida"] = "E99"           # estação fora do cadastro
    for _ in range(4):                                      # viagens de 05/06 que só chegam depois
        numero += 1
        inicio = datetime(2026, 6, 5, 21, sorteio.randint(0, 59), tzinfo=FUSO)
        atrasadas.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": "E07", "estacao_chegada": "E01",
                          "inicio": inicio.isoformat(), "fim": (inicio + timedelta(minutes=12)).isoformat()})
    return viagens, atrasadas


VIAGENS, _ATRASADAS = _gerar()
_estado = {"atrasadas_chegaram": False, "falha_temporaria": True}


def chegam_viagens_atrasadas():
    """Simula o dia seguinte: a fonte recebe 4 viagens de 05/06 que ainda não tinham sido enviadas."""
    if not _estado["atrasadas_chegaram"]:
        VIAGENS["2026-06-05"] = sorted(VIAGENS["2026-06-05"] + _ATRASADAS, key=lambda v: v["inicio"])
        _estado["atrasadas_chegaram"] = True


class _Api(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def _responder(self, status, corpo=None):
        self.send_response(status)
        if corpo is None:
            self.end_headers()
            return
        dados = json.dumps(corpo, ensure_ascii=False).encode("utf-8")
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def do_GET(self):
        url = urlparse(self.path)
        args = {k: v[0] for k, v in parse_qs(url.query).items()}
        if url.path == "/estacoes":
            return self._responder(200, ESTACOES)
        if url.path != "/viagens":
            return self._responder(404, {"erro": "caminho inexistente"})
        try:
            dia = date.fromisoformat(args.get("data", ""))
            pagina = int(args.get("pagina", "1"))
        except ValueError:
            return self._responder(400, {"erro": "use data=AAAA-MM-DD e pagina=1, 2, ..."})
        lista = VIAGENS.get(dia.isoformat(), [])
        if not lista:
            return self._responder(204)
        if dia.isoformat() == "2026-06-02" and pagina == 2 and _estado["falha_temporaria"]:
            _estado["falha_temporaria"] = False
            return self._responder(503, {"erro": "servidor ocupado, tente de novo"})
        paginas = -(-len(lista) // POR_PAGINA)
        inicio = (pagina - 1) * POR_PAGINA
        trecho = lista[inicio:inicio + POR_PAGINA]
        if dia.isoformat() == "2026-06-03" and pagina == 2:
            trecho = [lista[inicio - 1]] + trecho               # a página 2 repete o último da página 1
        self._responder(200, {"data": dia.isoformat(), "pagina": pagina, "total_paginas": paginas,
                              "total_registros": len(lista), "viagens": trecho})


_servidor = ThreadingHTTPServer(("127.0.0.1", 0), _Api)
threading.Thread(target=_servidor.serve_forever, daemon=True).start()
API = f"http://127.0.0.1:{_servidor.server_port}"
print(f"Cenário pronto: {len(ESTACOES)} estações, {sum(len(v) for v in VIAGENS.values())} viagens "
      f"de {PRIMEIRO_DIA:%d/%m} a {ULTIMO_DIA:%d/%m}. API em {API}")

Esta célula refaz a coleta da aula anterior e deixa a pasta `bruto/` pronta. Assim este notebook funciona sozinho.

In [ ]:
import time

import httpx


def pedir(caminho, parametros=None, tentativas=3):
    """Faz um pedido à API. Tenta de novo quando o erro é do servidor (5xx)."""
    for tentativa in range(1, tentativas + 1):
        resposta = httpx.get(API + caminho, params=parametros, timeout=10)
        if resposta.status_code == 204:
            return None                      # dia sem viagens: a resposta vem vazia
        if resposta.status_code >= 500 and tentativa < tentativas:
            print(f"  {resposta.status_code} na tentativa {tentativa}: nova tentativa em 1 segundo")
            time.sleep(1)
            continue
        resposta.raise_for_status()          # 4xx: o pedido está errado; insistir não resolve
        return resposta.json()


def coletar_dia(data):
    """Todas as viagens de um dia, página por página, e o total que a API informa."""
    viagens, pagina, total_paginas, total_registros = [], 1, 1, 0
    while pagina <= total_paginas:
        corpo = pedir("/viagens", {"data": data, "pagina": pagina})
        if corpo is None:                    # 204: não há viagens neste dia
            return [], 0
        viagens.extend(corpo["viagens"])
        total_paginas, total_registros = corpo["total_paginas"], corpo["total_registros"]
        pagina += 1
    return viagens, total_registros


import json
from datetime import datetime
from pathlib import Path

PASTA = Path("bruto")


def salvar_bruto(data, viagens, total_registros):
    """Guarda o dia exatamente como veio da API, com a hora da coleta."""
    PASTA.mkdir(exist_ok=True)
    arquivo = PASTA / f"viagens_{data}.json"
    conteudo = {"data": data, "coletado_em": datetime.now().isoformat(timespec="seconds"),
                "total_registros": total_registros, "viagens": viagens}
    arquivo.write_text(json.dumps(conteudo, ensure_ascii=False), encoding="utf-8")
    return arquivo


DIAS = [f"2026-06-0{d}" for d in range(1, 8)]   # de 01/06 a 07/06


for data in DIAS:
    salvar_bruto(data, *coletar_dia(data))
print(f"Pasta bruto/ pronta: {len(list(PASTA.glob('*.json')))} arquivos")

##  Passo 1 de 4 · Criar o banco e a tabela

O DuckDB é um banco de dados SQL que roda dentro do Python e guarda tudo num arquivo. A tabela `raw_viagens` tem as colunas da viagem, o dia de referência e a hora da coleta.

In [ ]:
# Rode e observe
import duckdb

con = duckdb.connect("pedala.duckdb")       # o banco é este arquivo
con.execute("""
    create table if not exists raw_viagens (
        data_ref        varchar,
        id_viagem       varchar,
        id_usuario      varchar,
        estacao_saida   varchar,
        estacao_chegada varchar,
        inicio          varchar,
        fim             varchar,
        coletado_em     varchar
    )
""")
con.execute("describe raw_viagens").df()

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Por que tudo é texto</b><br>A tabela bruta guarda as colunas como texto (<code>varchar</code>), do jeito que vieram. Converter datas e números é trabalho da transformação. Assim, um valor estranho na fonte não impede a carga: ele chega, fica registrado e é tratado depois.</div>

##  Passo 2 de 4 · Gravar, e gravar de novo

A função mais simples possível: lê o arquivo e insere as linhas. Rode a carga dos sete dias **duas vezes**.

In [ ]:
# Rode e observe
def linhas_do_arquivo(arquivo):
    """Lê um arquivo da pasta bruto/ e devolve as linhas da tabela raw_viagens."""
    conteudo = json.loads(arquivo.read_text(encoding="utf-8"))
    return conteudo["data"], [
        (conteudo["data"], v["id_viagem"], v["id_usuario"], v["estacao_saida"],
         v["estacao_chegada"], v["inicio"], v["fim"], conteudo["coletado_em"])
        for v in conteudo["viagens"]
    ]


def gravar_dia_ingenuo(con, arquivo):
    data, linhas = linhas_do_arquivo(arquivo)
    con.executemany("insert into raw_viagens values (?, ?, ?, ?, ?, ?, ?, ?)", linhas)


for rodada in (1, 2):
    for arquivo in sorted(PASTA.glob("*.json")):
        gravar_dia_ingenuo(con, arquivo)
    print(f"Depois da rodada {rodada}:", con.execute("select count(*) from raw_viagens").fetchone()[0], "linhas")

<div style="border-left:5px solid #F9006F;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#F9006F">A carga somou em vez de substituir</b><br>Na segunda rodada, cada viagem entrou de novo. Em um pipeline isso acontece toda vez que alguém reexecuta um dia, por falha ou por correção. Uma carga que pode rodar várias vezes com o mesmo resultado se chama <b>idempotente</b>.</div>

##  Passo 3 de 4 · Substituir o dia

A correção: antes de inserir um dia, apague o que já existe daquele dia. As duas operações ficam numa **transação** (`begin` … `commit`): ou as duas acontecem, ou nenhuma.

In [ ]:
# Rode e observe
con.execute("delete from raw_viagens")      # recomeça do zero, sem a duplicação de antes


def gravar_dia(con, arquivo):
    """Substitui o dia inteiro: apaga o que havia daquele dia e grava de novo, numa transação."""
    data, linhas = linhas_do_arquivo(arquivo)
    con.execute("begin")
    try:
        con.execute("delete from raw_viagens where data_ref = ?", [data])
        con.executemany("insert into raw_viagens values (?, ?, ?, ?, ?, ?, ?, ?)", linhas)
        con.execute("commit")
    except Exception:
        con.execute("rollback")              # desfaz o delete: o dia antigo volta inteiro
        raise
    return len(linhas)


for rodada in (1, 2):
    for arquivo in sorted(PASTA.glob("*.json")):
        gravar_dia(con, arquivo)
    print(f"Depois da rodada {rodada}:", con.execute("select count(*) from raw_viagens").fetchone()[0], "linhas")
con.execute("select data_ref, count(*) as linhas from raw_viagens group by data_ref order by data_ref").df()

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Leia assim</b><br>624 linhas para 623 viagens: a viagem repetida pela API em 03/06 continua aqui, porque a tabela bruta é um espelho da fonte. Rodar de novo não muda nada: a carga agora é idempotente.</div>

### Sua vez

Grave também o cadastro das estações, na tabela `raw_estacoes`, de modo que rodar duas vezes **não duplique**. Como o cadastro é pequeno, a forma mais simples é substituir a tabela inteira a cada carga: `create or replace table` faz isso. Use `pedir("/estacoes")` para buscar os dados.

In [ ]:
# Sua vez
def gravar_estacoes(con):
    return None   # crie a tabela, insira as estações e devolva quantas gravou

In [ ]:
# Confira
gravadas = gravar_estacoes(con)
gravar_estacoes(con)                         # de novo: não pode duplicar
linhas = con.execute("select count(*) from raw_estacoes").fetchone()[0] if gravadas is not None else None
conferir(gravadas, linhas == 8, "8 estações, mesmo depois de gravar duas vezes",
         f"A tabela tem {linhas} linhas depois de duas cargas; o esperado é 8")

<details>
<summary><b>Ver uma resposta</b> (tente antes)</summary>

```python
def gravar_estacoes(con):
    """O cadastro é pequeno: a cada carga, substitui a tabela inteira."""
    estacoes = pedir("/estacoes")
    con.execute("create or replace table raw_estacoes (id_estacao varchar, nome varchar, bairro varchar, vagas varchar)")
    con.executemany("insert into raw_estacoes values (?, ?, ?, ?)",
                    [(e["id_estacao"], e["nome"], e["bairro"], str(e["vagas"])) for e in estacoes])
    return len(estacoes)
```
</details>

##  Passo 4 de 4 · Fechar e abrir de novo

O dado está no arquivo `pedala.duckdb`, e não na memória do notebook. Feche a conexão, abra outra e conte.

In [ ]:
# Rode e observe
con.close()
con = duckdb.connect("pedala.duckdb")
print(con.execute("select count(*) from raw_viagens").fetchone()[0], "linhas continuam no arquivo")
con.close()

## O que você aprendeu

- A tabela bruta guarda texto, como veio da fonte, com o dia de referência e a hora da coleta.
- Inserir sem pensar duplica na segunda execução.
- Apagar o dia e inserir de novo, numa transação, torna a carga **idempotente**.
- Um cadastro pequeno pode ser substituído inteiro a cada carga.

**Próxima aula:** [Prática: transformar em camadas](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-fundamentos-camadas.md).